# LSTM with Confidence-Boosted Gate (CBG)
### Research notebook — optimized from Class A

**What changed and why:**
- `nn.Linear(..., bias=False)` + separate `nn.Parameter` bias → standard `nn.Linear` (bias=True). Cleaner, same math, fewer parameters to manage.
- `confidence_layer` now takes only `[h_t ∥ c_t]` (2×hidden) instead of `[x_t ∥ h_t ∥ c_t]`. The input x is already baked into h via the gates — passing it again adds redundancy and makes the gate input-size dependent (breaks if you change embedding dim).
- `boost_factor` renamed `boost_proj` to match its role as a projection.
- Gate helper methods combined into `lstm_cell` directly (one less indirection, easier to trace gradients mentally).
- `forward` returns `(outputs, h, c)` so you can do multi-turn inference or teacher forcing without re-running.
- Training loop clips gradients — fixes the loss spike you saw around epoch 73.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from rich.progress import track

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
text = """the young boy walked down the long road that led to the old village at the end of the valley the road was dusty and dry and the sun was high in the sky the boy carried a small brown bag on his back inside the bag was a loaf of bread a bottle of water and a letter from his mother the letter was for the old man who lived alone at the edge of the village the old man had been a farmer for most of his life and knew every tree and stone in the valley the boy had never met the old man but his mother said he was kind and wise and always had a story to tell the boy walked past a wide green field where some cows were eating grass slowly near a wooden fence the fence was old and some parts had fallen down a black dog ran along the fence barking at the cows but the cows did not care and kept eating the boy stopped to watch the dog for a moment and then continued walking down the road the wind began to blow softly and the tall grass on both sides of the road moved gently like waves on a calm sea the boy looked up at the sky and saw a few white clouds moving slowly toward the mountains in the distance the mountains were tall and dark and always had snow on their peaks even in summer as the boy got closer to the village he could hear the sound of water a small river ran through the middle of the village and the people had built a stone bridge over it long ago children were playing near the river throwing stones into the water and laughing the boy crossed the bridge and looked down at the clear water below he could see small fish swimming near the bottom of the river moving in and out of the shadows the village had about twenty small houses most of them made of stone with red or brown roofs a few old trees grew between the houses giving shade to the narrow streets some women were sitting outside their doors talking quietly while their children played nearby an old man was fixing a wooden cart near the well in the center of the village the boy asked him where the farmer lived and the old man pointed to a small house at the far end of the village near a large oak tree the boy thanked him and walked toward the house when he arrived he knocked on the wooden door softly after a moment the door opened and a tall old man with white hair and kind eyes looked down at him the boy held out the letter and said it was from his mother the old man smiled and took the letter and invited the boy inside the house was small but warm a fire was burning in the corner and the smell of soup filled the room the old man read the letter slowly by the light of the fire and then folded it carefully and put it in his coat pocket he thanked the boy and offered him a bowl of soup and a place to rest the boy sat down near the fire and the old man began to tell him a story about the valley long ago"""

words = text.lower().split()

# Deduplicated vocabulary (original code had duplicate keys — wti only keeps last occurrence)
vocab = list(dict.fromkeys(words))  # preserves order, removes duplicates
wti = {w: i for i, w in enumerate(vocab)}
itw = {i: w for i, w in enumerate(vocab)}

seq = 25
X = torch.tensor([[wti[w] for w in words[i:i+seq]] for i in range(len(words) - seq)])
y = torch.tensor([[wti[words[i+seq]]] for i in range(len(words) - seq)])

print(f'Vocab size: {len(vocab)}  |  Sequences: {len(X)}')

## The LSTM + CBG model

### How the Confidence-Boosted Gate works

After computing `h_t` normally from the four standard LSTM gates, CBG asks:
> *"How confident are we in this hidden state?"*

It does this by looking at `[h_t ∥ c_t]` — both the hidden state and the cell memory — and producing a scalar confidence score between 0 and 1.

- If confidence is **high (≈1)**: the LSTM was already doing well, leave `h_t` alone.
- If confidence is **low (≈0)**: the gate applies a learned boost to correct or amplify the hidden state.

```
conf   = σ( W_conf · [h_t ∥ c_t] )
boost  = W_boost · h_t
h_out  = h_t + (1 − conf) · boost
```

This is inspired by residual connections and mixture-of-experts gating. When the model is uncertain, it can route through a learned correction rather than passing a weak `h_t` downstream.

In [ ]:
class LSTMwithCBG(nn.Module):
    """
    Standard LSTM cell extended with a Confidence-Boosted Gate (CBG).

    The CBG sits after the standard h_t computation and modulates
    the output hidden state based on how 'confident' the cell state
    is. When confidence is low, a learned boost is added to h_t.
    """

    def __init__(self, input_size: int, hidden_size: int):
        super().__init__()
        self.hidden_size = hidden_size

        # ── Standard LSTM gates (bias=True is the default, cleaner than separate params) ──
        self.forget_gate  = nn.Linear(input_size + hidden_size, hidden_size)
        self.input_gate   = nn.Linear(input_size + hidden_size, hidden_size)
        self.cell_gate    = nn.Linear(input_size + hidden_size, hidden_size)
        self.output_gate  = nn.Linear(input_size + hidden_size, hidden_size)

        # ── Confidence-Boosted Gate ──
        # Takes [h_t ∥ c_t] — not x_t, since x is already encoded in h via the gates
        self.conf_gate  = nn.Linear(hidden_size * 2, 1)
        self.boost_proj = nn.Linear(hidden_size, hidden_size)

        self._init_weights()

    def _init_weights(self):
        """Orthogonal init on recurrent weights reduces vanishing/exploding gradients."""
        for name, param in self.named_parameters():
            if 'weight' in name:
                nn.init.orthogonal_(param)
            elif 'bias' in name:
                nn.init.zeros_(param)
        # Forget gate bias = 1 is a known trick to encourage remembering at init
        nn.init.ones_(self.forget_gate.bias)

    def lstm_cell(
        self,
        x: torch.Tensor,       # (batch, input_size)
        h_prev: torch.Tensor,  # (batch, hidden_size)
        c_prev: torch.Tensor,  # (batch, hidden_size)
    ):
        combined = torch.cat([x, h_prev], dim=-1)  # (batch, input+hidden)

        f = torch.sigmoid(self.forget_gate(combined))
        i = torch.sigmoid(self.input_gate(combined))
        g = torch.tanh(self.cell_gate(combined))
        o = torch.sigmoid(self.output_gate(combined))

        c_t = f * c_prev + i * g
        h_t = o * torch.tanh(c_t)

        # ── Confidence-Boosted Gate ──
        conf  = torch.sigmoid(self.conf_gate(torch.cat([h_t, c_t], dim=-1)))  # (batch, 1)
        boost = self.boost_proj(h_t)                                            # (batch, hidden)
        h_out = h_t + (1.0 - conf) * boost

        return h_out, c_t

    def forward(
        self,
        x: torch.Tensor,         # (seq_len, batch, input_size)
        h0: torch.Tensor = None,
        c0: torch.Tensor = None,
    ):
        seq_len, batch, _ = x.shape
        h = h0 if h0 is not None else torch.zeros(batch, self.hidden_size, device=x.device)
        c = c0 if c0 is not None else torch.zeros(batch, self.hidden_size, device=x.device)

        outputs = []
        for t in range(seq_len):
            h, c = self.lstm_cell(x[t], h, c)
            outputs.append(h.unsqueeze(0))

        return torch.cat(outputs, dim=0), h, c  # (seq_len, batch, hidden), h, c

In [ ]:
EMBED_DIM  = 100
HIDDEN_DIM = 128  # slightly larger hidden — CBG adds expressive capacity

embedding  = nn.Embedding(len(vocab), EMBED_DIM).to(device)
model      = LSTMwithCBG(EMBED_DIM, HIDDEN_DIM).to(device)
classifier = nn.Linear(HIDDEN_DIM, len(vocab)).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'LSTMwithCBG parameters: {total_params:,}')

In [ ]:
loss_fn   = nn.CrossEntropyLoss()
all_params = list(embedding.parameters()) + list(model.parameters()) + list(classifier.parameters())
optimizer  = optim.Adam(all_params, lr=0.005, weight_decay=1e-5)  # weight_decay regularizes

# Cosine LR schedule — gently reduces LR instead of keeping it flat
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100, eta_min=1e-5)

dataset    = torch.utils.data.TensorDataset(X, y)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=32, shuffle=True, drop_last=True)

In [ ]:
EPOCHS     = 100
CLIP_NORM  = 1.0  # gradient clipping — prevents the loss spike seen in Class A

for epoch in track(range(EPOCHS), description='Training: '):
    model.train()
    epoch_loss = 0.0
    for X_batch, y_batch in dataloader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        emb    = embedding(X_batch)                   # (batch, seq, embed)
        out, _, _ = model(emb.permute(1, 0, 2))       # (seq, batch, hidden)
        logits = classifier(out[-1])                  # last step → (batch, vocab)
        loss   = loss_fn(logits, y_batch.squeeze())

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(all_params, CLIP_NORM)  # <-- kills spikes
        optimizer.step()

        epoch_loss = loss.item()

    scheduler.step()
    print(f'loss: {epoch_loss:.6f}')

In [ ]:
def generate(
    prompt: str,
    max_tokens: int = 50,
    temperature: float = 0.8,
    top_p: float = 0.75,
) -> str:
    model.eval()
    words_out  = prompt.lower().split()
    curr_words = list(words_out)

    with torch.no_grad():
        for _ in range(max_tokens):
            window = curr_words[-seq:]
            ids    = torch.tensor([[wti[w.strip()] for w in window]], device=device)
            emb    = embedding(ids)                        # (1, seq, embed)
            out, _, _ = model(emb.permute(1, 0, 2))       # (seq, 1, hidden)
            logits = classifier(out[-1])                   # (1, vocab)

            probs = F.softmax(logits / temperature, dim=-1)

            # Nucleus (top-p) sampling
            sorted_p, sorted_idx = torch.sort(probs, descending=True)
            cum_p = torch.cumsum(sorted_p, dim=-1)
            keep  = (cum_p - sorted_p) < top_p
            sorted_p[~keep] = 0.0
            sorted_p /= sorted_p.sum()
            probs = torch.zeros_like(probs).scatter_(-1, sorted_idx, sorted_p)

            word = itw[torch.multinomial(probs, 1).item()]
            curr_words.append(word)
            words_out.append(word)

    return ' '.join(words_out)

print(generate('the old man', max_tokens=40))